# TP Jour 3 — Spark SQL Avancé, Rigueur Statistique & Couche Gold

### Objectifs de la séance
1. Diagnostiquer l'**asymétrie (`SKEWNESS`)** de vos variables quantitatives et justifier l'emploi d'indicateurs robustes (**Médiane `PERCENTILE_APPROX`**, **IQR `P75 - P25`**, **P90**) plutôt que la seule moyenne arithmétique.
2. Écrire des requêtes **Spark SQL avancées** mobilisant des CTE (`WITH ...`) et des **fonctions de fenêtrage** (`DENSE_RANK`, `ROW_NUMBER`, cumul de **Pareto**, variation temporelle **`LAG`**).
3. Construire au moins **2 à 3 tables agrégées métier dans `/lakehouse/gold/`** prêtes à alimenter votre Dashboard Streamlit à faible latence.
4. Éviter le *Small Files Problem* sur HDFS et compacter votre table Silver avec `OPTIMIZE`.

**Problématique du projet**

Prédire, au moment de la livraison, la probabilité qu’un client écrive un commentaire après livraison. La cible Silver `a_un_commentaire` vaut 1 si un message est écrit à partir de la livraison réelle, et 0 sinon. Le périmètre est celui des commandes livrées à un seul article.

Les tables Gold résument les taux observés par catégorie, par mois de livraison et par couple prix–retard. Ces taux décrivent l’échantillon historique. Lors de l’entraînement du futur modèle, les agrégats utilisés comme variables devront être calculés sur l’ensemble d’entraînement pour éviter d’utiliser les réponses du jeu de test.

**Exécution**

Dans VS Code, sélectionner le serveur Jupyter existant `http://localhost:8888/?token=bigdata` et son kernel `Python 3 (ipykernel)`. Dans JupyterLab, utiliser ce kernel puis exécuter les cellules dans l’ordre.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from delta.tables import DeltaTable

spark = SparkSession.builder.appName("J3_Analytics_SQL_Gold").getOrCreate()
SILVER_MAIN_PATH = "/lakehouse/silver/olist_commentaires_livraison"

# Fixer un instantané pour que les trois tables Gold utilisent les mêmes données.
silver_version = DeltaTable.forPath(spark, SILVER_MAIN_PATH).history(1).first().version
silver_source = spark.read.format("delta").option("versionAsOf", silver_version).load(SILVER_MAIN_PATH)
silver_source.createOrReplaceTempView("v_silver")

source_summary = spark.sql('''
    SELECT COUNT(*) AS nb_commandes,
           COUNT(DISTINCT order_id) AS commandes_uniques,
           SUM(a_un_commentaire) AS nb_commentaires,
           SUM(CASE WHEN a_un_commentaire IS NULL OR a_un_commentaire NOT IN (0, 1)
                    OR date_livraison IS NULL OR categorie_produit IS NULL
                    OR prix IS NULL OR prix < 0 OR retard_jours IS NULL OR retard_jours < 0
                    THEN 1 ELSE 0 END) AS lignes_invalides
    FROM v_silver
''').first()
assert source_summary.nb_commandes > 0, "La table Silver est vide."
assert source_summary.nb_commandes == source_summary.commandes_uniques, "order_id doit être unique."
assert source_summary.lignes_invalides == 0, "Vérifie les valeurs de la table Silver dans J2."
print(f"Spark {spark.version} | Silver version {silver_version}")
print(f"Commandes : {source_summary.nb_commandes:,} | Commentaires après livraison : {source_summary.nb_commentaires:,}")
print(f"Taux global observé : {source_summary.nb_commentaires / source_summary.nb_commandes:.1%}")

:: loading settings :: url = jar:file:/opt/venv/lib/python3.15/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
io.delta#delta-spark_4.2_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-bc66f034-2897-4876-9e0e-1e23da7b320e;1.0
	confs: [default]
	found io.delta#delta-spark_4.2_2.13;4.4.1 in central
	found io.delta#delta-storage;4.4.1 in central
	found io.unitycatalog#unitycatalog-client;0.6.0 in central
	found org.slf4j#slf4j-api;2.0.13 in central
	found org.apache.logging.log4j#log4j-slf4j2-impl;2.25.3 in central
	found org.apache.logging.log4j#log4j-api;2.25.3 in central
	found com.google.code.findbugs#jsr305;3.0.2 in central
	found io.unitycatalog#unitycatalog-hadoop;0.6.0 in central
	found org.apache.logging.log4j#log4j-core;2.25.3 in central
	found io.delta#delta-kernel-api;4.4.1 in central
	found

Spark 4.2.0 | Silver version 7
Commandes : 86,835 | Commentaires après livraison : 31,286
Taux global observé : 36.0%


## 1. Analyse d'Asymétrie : Moyenne vs Médiane & Écart Interquartile (IQR)

### Exercice 1.1 — Étudier la distribution de votre métrique quantitative principale
Écrivez une requête Spark SQL sur `v_silver` calculant sur votre variable continue principale :
- `COUNT(*)`, `AVG(...)`, `STDDEV(...)`, `SKEWNESS(...)`
- Les quantiles `P25`, `P50` (médiane), `P75`, `P90`, `P95` via `PERCENTILE_APPROX(col, q)` et `MAX(...)`.

Rédigez ensuite en Markdown **votre interprétation statistique** (écart moyenne/médiane, impact des valeurs extrêmes, choix de l'indicateur retenu pour votre dashboard).

In [2]:
sql_distribution = '''
WITH mesures AS (
    SELECT 'prix_brl' AS variable, prix AS valeur FROM v_silver
    UNION ALL
    SELECT 'frais_livraison_brl', frais_livraison FROM v_silver
    UNION ALL
    SELECT 'delai_livraison_jours', CAST(delai_livraison_jours AS DOUBLE) FROM v_silver
), resume AS (
    SELECT variable, COUNT(*) AS nb_commandes, COUNT(valeur) AS nb_renseignees,
           AVG(valeur) AS moyenne, STDDEV(valeur) AS ecart_type, SKEWNESS(valeur) AS asymetrie,
           PERCENTILE_APPROX(valeur, ARRAY(0.25, 0.5, 0.75, 0.9, 0.95), 10000) AS quantiles,
           MAX(valeur) AS maximum
    FROM mesures GROUP BY variable
)
SELECT variable, nb_commandes, nb_renseignees, moyenne, ecart_type, asymetrie,
       quantiles[0] AS p25, quantiles[1] AS mediane, quantiles[2] AS p75,
       quantiles[2] - quantiles[0] AS iqr, quantiles[3] AS p90, quantiles[4] AS p95, maximum
FROM resume ORDER BY variable
'''
df_dist_summary = spark.sql(sql_distribution)
df_dist_summary.show(truncate=False)

+---------------------+------------+--------------+------------------+------------------+------------------+-----+-------+-----+-----------------+-----+-----+-------+
|variable             |nb_commandes|nb_renseignees|moyenne           |ecart_type        |asymetrie         |p25  |mediane|p75  |iqr              |p90  |p95  |maximum|
+---------------------+------------+--------------+------------------+------------------+------------------+-----+-------+-----+-----------------+-----+-----+-------+
|delai_livraison_jours|86835       |86835         |12.577658778142455|9.620642007257043 |3.7753992298984933|7.0  |10.0   |16.0 |9.0              |23.0 |30.0 |210.0  |
|frais_livraison_brl  |86835       |86835         |20.328848275466537|15.91722747820867 |5.578419988198597 |13.43|16.46  |21.32|7.890000000000001|34.22|46.0 |409.68 |
|prix_brl             |86835       |86835         |129.69834605872984|196.61804306556976|7.70482746078822  |44.0 |79.9   |144.0|100.0            |249.9|379.9|6735.0 

**Interprétation statistique sur le périmètre Olist**

Le prix moyen vaut environ **129,70 BRL**, contre une médiane de **79,90 BRL**. L’asymétrie positive d’environ **7,70** reflète une longue queue vers les articles chers. Le maximum atteint **6 735 BRL**. Les quantiles approximatifs Q1 = 44,00 BRL et Q3 = 143,99 BRL donnent un IQR d’environ 99,99 BRL.

Pour décrire le prix typique d’une commande, le dashboard affichera la **médiane**, l’**IQR** et le **P90** à côté du taux de commentaire et de l’effectif. Les valeurs extrêmes sont conservées pour l’analyse. La comparaison du prix entre commandes avec et sans commentaire pourra alimenter les hypothèses du Jour 4.

Les quantiles sont approximatifs et peuvent varier légèrement selon l’exécution distribuée.

---
## 2. Table Gold #1 : Classement Métier & Loi de Pareto (`OVER ORDER BY ...`)

### Exercice 2.1 — Agrégation par segment/catégorie + Courbe cumulée de Pareto
Construisez en Spark SQL une première table Gold agrégée par une dimension métier pertinente de votre dataset incluant :
- Le volume d'observations, le total et les indicateurs robustes (**médiane**, **IQR**).
- Le rang via `DENSE_RANK() OVER (ORDER BY ... DESC)`.
- Le pourcentage cumulé du total (**analyse de Pareto 80/20**) via :
  `SUM(metric_total) OVER (ORDER BY metric_total DESC ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) / SUM(metric_total) OVER ()`.

Sauvegardez-la avec `.coalesce(1)` au format **Delta** dans `/lakehouse/gold/<nom_gold_1>`.

**Application Olist**

Le Pareto classe les catégories selon leur nombre de commentaires écrits après livraison. La part cumulée permet de mesurer où se concentrent les commentaires. Le taux de commentaire et l’effectif sont affichés à côté du volume. Le seuil de 80 % est mesuré sur les données ; la règle 80/20 reste une hypothèse à examiner.

In [3]:
GOLD_PARETO_PATH = "/lakehouse/gold/olist_commentaires_categories"

sql_pareto = '''
WITH categories AS (
    SELECT categorie_produit, COUNT(*) AS nb_commandes,
           SUM(a_un_commentaire) AS nb_commentaires,
           AVG(CAST(a_un_commentaire AS DOUBLE)) AS taux_commentaire,
           SUM(prix) AS total_prix_brl,
           PERCENTILE_APPROX(prix, ARRAY(0.25, 0.5, 0.75), 10000) AS quantiles_prix,
           PERCENTILE_APPROX(delai_livraison_jours, 0.5, 10000) AS delai_median_jours
    FROM v_silver GROUP BY categorie_produit
)
SELECT categorie_produit, nb_commandes, nb_commentaires, taux_commentaire, total_prix_brl,
       quantiles_prix[1] AS prix_median_brl,
       quantiles_prix[2] - quantiles_prix[0] AS prix_iqr_brl,
       delai_median_jours,
       DENSE_RANK() OVER (ORDER BY nb_commentaires DESC) AS rang_volume_commentaires,
       SUM(nb_commentaires) OVER (
           ORDER BY nb_commentaires DESC, categorie_produit
           ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
       ) / NULLIF(SUM(nb_commentaires) OVER (), 0) AS part_commentaires_cumulee,
       SUM(nb_commandes) OVER (
           ORDER BY nb_commentaires DESC, categorie_produit
           ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
       ) / SUM(nb_commandes) OVER () AS part_commandes_cumulee,
       nb_commandes < 100 AS effectif_faible
FROM categories
'''
df_gold_1 = spark.sql(sql_pareto)
df_gold_1.coalesce(1).write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(GOLD_PARETO_PATH)
df_gold_1.orderBy(F.desc("nb_commentaires"), "categorie_produit").show(15, truncate=False)

pareto_rows = df_gold_1.orderBy(F.desc("nb_commentaires"), "categorie_produit").collect()
if source_summary.nb_commentaires:
    nb_categories_80 = next(i for i, row in enumerate(pareto_rows, 1) if row.part_commentaires_cumulee >= 0.8)
    print(f"80 % des commentaires proviennent de {nb_categories_80} catégories sur {len(pareto_rows)}.")
print("Le rang décrit le volume de commentaires. Le taux permet de comparer les fréquences entre catégories.")

26/11/19 07:45:17 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 07:45:17 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 07:45:17 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 07:45:17 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 07:45:17 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 07:45:17 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 0

+----------------------+------------+---------------+-------------------+------------------+---------------+------------------+------------------+------------------------+-------------------------+----------------------+---------------+
|categorie_produit     |nb_commandes|nb_commentaires|taux_commentaire   |total_prix_brl    |prix_median_brl|prix_iqr_brl      |delai_median_jours|rang_volume_commentaires|part_commentaires_cumulee|part_commandes_cumulee|effectif_faible|
+----------------------+------------+---------------+-------------------+------------------+---------------+------------------+------------------+------------------------+-------------------------+----------------------+---------------+
|cama_mesa_banho       |7836        |3118           |0.39790709545686576|776075.3500000468 |81.8           |69.91             |11                |1                       |0.0996611903087643       |0.09024011055449992   |false          |
|beleza_saude          |7976        |2663           

26/11/19 07:45:21 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 07:45:21 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 07:45:21 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 07:45:21 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 07:45:21 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 07:45:21 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 0

80 % des commentaires proviennent de 16 catégories sur 74.
Le rang décrit le volume de commentaires. Le taux permet de comparer les fréquences entre catégories.


## 3. Table Gold #2 : Dynamique Temporelle & Croissance avec `LAG()`

### Exercice 3.1 — Série temporelle et taux d'évolution période-sur-période
Agrégez votre table `v_silver` selon son axe temporel (jour, semaine, mois ou année selon votre sujet) et utilisez la fonction de fenêtrage **`LAG(metric, 1) OVER (ORDER BY periode)`** (ou une moyenne glissante `AVG(...) OVER (ORDER BY periode ROWS BETWEEN ... PRECEDING AND CURRENT ROW)`) pour mesurer la dynamique d'évolution.

Sauvegardez le résultat dans `/lakehouse/gold/<nom_gold_2>`.

**Application Olist**

La période est le mois de livraison. Le calendrier inclut les mois sans commande. La variation du volume de commentaires est exprimée en pourcentage ; la variation du taux est exprimée en points de pourcentage. Les divisions par un volume nul donnent NULL et les mois de moins de 100 commandes sont signalés. Le mois correspond à une cohorte de livraisons, même si les commentaires ont été écrits plus tard. La fin de période offre moins de recul pour observer les réponses des clients.

In [4]:
GOLD_TIME_PATH = "/lakehouse/gold/olist_commentaires_mensuels"

sql_temps = '''
WITH mensuel AS (
    SELECT TRUNC(TO_DATE(date_livraison), 'MM') AS periode,
           COUNT(*) AS nb_commandes, SUM(a_un_commentaire) AS nb_commentaires,
           PERCENTILE_APPROX(prix, 0.5, 10000) AS prix_median_brl,
           PERCENTILE_APPROX(delai_livraison_jours, 0.5, 10000) AS delai_median_jours
    FROM v_silver GROUP BY TRUNC(TO_DATE(date_livraison), 'MM')
), calendrier AS (
    SELECT EXPLODE(SEQUENCE(MIN(periode), MAX(periode), INTERVAL 1 MONTH)) AS periode
    FROM mensuel
), serie AS (
    SELECT c.periode, COALESCE(m.nb_commandes, 0) AS nb_commandes,
           COALESCE(m.nb_commentaires, 0) AS nb_commentaires,
           m.nb_commentaires / NULLIF(m.nb_commandes, 0) AS taux_commentaire,
           m.prix_median_brl, m.delai_median_jours
    FROM calendrier c LEFT JOIN mensuel m ON c.periode = m.periode
), decalee AS (
    SELECT *,
           LAG(nb_commandes, 1) OVER (ORDER BY periode) AS commandes_mois_precedent,
           LAG(nb_commentaires, 1) OVER (ORDER BY periode) AS commentaires_mois_precedent,
           LAG(taux_commentaire, 1) OVER (ORDER BY periode) AS taux_mois_precedent
    FROM serie
)
SELECT *,
       100.0 * (nb_commentaires - commentaires_mois_precedent)
           / NULLIF(commentaires_mois_precedent, 0) AS evolution_commentaires_pct,
       100.0 * (taux_commentaire - taux_mois_precedent) AS variation_taux_points,
       nb_commandes < 100 AS effectif_faible
FROM decalee
'''
df_gold_2 = spark.sql(sql_temps)
df_gold_2.coalesce(1).write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(GOLD_TIME_PATH)
df_gold_2.orderBy("periode").show(30, truncate=False)

26/11/19 07:45:23 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 07:45:23 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 07:45:23 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 07:45:23 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 07:45:23 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 07:45:23 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 0

+----------+------------+---------------+--------------------+---------------+------------------+------------------------+---------------------------+--------------------+--------------------------+---------------------+---------------+
|periode   |nb_commandes|nb_commentaires|taux_commentaire    |prix_median_brl|delai_median_jours|commandes_mois_precedent|commentaires_mois_precedent|taux_mois_precedent |evolution_commentaires_pct|variation_taux_points|effectif_faible|
+----------+------------+---------------+--------------------+---------------+------------------+------------------------+---------------------------+--------------------+--------------------------+---------------------+---------------+
|2016-10-01|185         |123            |0.6648648648648648  |73.35          |13                |NULL                    |NULL                       |NULL                |NULL                      |NULL                 |false          |
|2016-11-01|48          |36             |0.75       

## 4. Table Gold #3 (Axe complémentaire libre) & Compaction `OPTIMIZE`

### Exercice 4.1 — Produire un 3e axe d'analyse (`ROW_NUMBER() OVER (PARTITION BY ...)`) et compacter Silver
1. Créez une 3e table Gold répondant à une question spécifique de votre problématique (ex. top sous-catégorie par région/groupe via `ROW_NUMBER() OVER (PARTITION BY groupe ORDER BY ... DESC) = 1`).
2. Exécutez `OPTIMIZE delta.'{SILVER_MAIN_PATH}'` et expliquez pourquoi la multiplication de micro-fichiers pénalise le NameNode HDFS.

**Application Olist**

Le troisième axe compare cinq tranches de prix en BRL à quatre niveaux de retard. `ROW_NUMBER()` classe les prix par taux observé au sein de chaque niveau de retard. Les groupes d’au moins 100 commandes sont prioritaires, puis le taux et l’effectif départagent les groupes. Toutes les combinaisons observées sont enregistrées pour le dashboard, avec leur rang.

**Interprétation de la compaction**

Chaque petit fichier impose au NameNode des métadonnées et au moteur des ouvertures de fichiers et des tâches supplémentaires. `OPTIMIZE` regroupe les fichiers Silver pour réduire ces coûts. Les tables Gold sont petites et écrites avec `coalesce(1)`. Un contrôle du nombre de lignes et de leur empreinte vérifie que la compaction conserve le contenu. Une table déjà compacte peut ne nécessiter aucune fusion.

In [5]:
GOLD_PRICE_DELAY_PATH = "/lakehouse/gold/olist_commentaires_prix_retard"

sql_prix_retard = '''
WITH tranches AS (
    SELECT *,
           CASE WHEN prix <= 20 THEN 0 WHEN prix <= 50 THEN 1
                WHEN prix <= 100 THEN 2 WHEN prix <= 200 THEN 3 ELSE 4 END AS ordre_prix,
           CASE WHEN retard_jours = 0 THEN 0 WHEN retard_jours <= 3 THEN 1
                WHEN retard_jours <= 7 THEN 2 ELSE 3 END AS ordre_retard
    FROM v_silver
), groupes AS (
    SELECT ordre_prix, ordre_retard, COUNT(*) AS nb_commandes,
           SUM(a_un_commentaire) AS nb_commentaires,
           AVG(CAST(a_un_commentaire AS DOUBLE)) AS taux_commentaire,
           PERCENTILE_APPROX(prix, 0.5, 10000) AS prix_median_brl,
           PERCENTILE_APPROX(delai_livraison_jours, 0.5, 10000) AS delai_median_jours
    FROM tranches GROUP BY ordre_prix, ordre_retard
), classes AS (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY ordre_retard
               ORDER BY CASE WHEN nb_commandes >= 100 THEN 0 ELSE 1 END,
                        taux_commentaire DESC, nb_commandes DESC, ordre_prix
           ) AS rang_taux_par_retard
    FROM groupes
)
SELECT *,
       CASE ordre_prix WHEN 0 THEN '0 à 20 BRL' WHEN 1 THEN ']20, 50] BRL'
            WHEN 2 THEN ']50, 100] BRL' WHEN 3 THEN ']100, 200] BRL'
            ELSE 'Plus de 200 BRL' END AS tranche_prix,
       CASE ordre_retard WHEN 0 THEN 'À temps' WHEN 1 THEN '1 à 3 jours'
            WHEN 2 THEN '4 à 7 jours' ELSE '8 jours et plus' END AS tranche_retard,
       nb_commandes < 100 AS effectif_faible
FROM classes
'''
df_gold_3 = spark.sql(sql_prix_retard)
df_gold_3.coalesce(1).write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(GOLD_PRICE_DELAY_PATH)
print("Tranches de prix au taux de commentaire le plus élevé, pour chaque niveau de retard (au moins 100 commandes)")
(
    df_gold_3.where((F.col("rang_taux_par_retard") == 1) & ~F.col("effectif_faible"))
    .orderBy("ordre_retard")
    .select("tranche_retard", "tranche_prix", "nb_commandes", "nb_commentaires", "taux_commentaire")
    .show(truncate=False)
)
print("Les autres tranches sont aussi enregistrées dans la table Gold pour comparer tous les prix à retard identique.")

# Contrôler que les trois agrégations couvrent toutes les commandes et tous les commentaires.
for path in (GOLD_PARETO_PATH, GOLD_TIME_PATH, GOLD_PRICE_DELAY_PATH):
    table = spark.read.format("delta").load(path)
    totals = table.agg(F.sum("nb_commandes").alias("n"), F.sum("nb_commentaires").alias("positifs")).first()
    assert totals.n == source_summary.nb_commandes and totals.positifs == source_summary.nb_commentaires, path
    print(f"[GOLD OK] {path} : {table.count()} lignes agrégées")

# Compacter Silver et vérifier l'intégrité des lignes avant/après.
def signature_silver(frame):
    return frame.agg(
        F.count("*").alias("n"),
        F.sum(F.xxhash64(*frame.columns).cast("decimal(38,0)")).alias("empreinte"),
    ).first()


delta_silver = DeltaTable.forPath(spark, SILVER_MAIN_PATH)
avant_optimize = delta_silver.toDF()
signature_avant = signature_silver(avant_optimize)
fichiers_avant = delta_silver.detail().select("numFiles").first().numFiles
spark.sql(f"OPTIMIZE delta.`{SILVER_MAIN_PATH}`").show(truncate=False)
apres_optimize = spark.read.format("delta").load(SILVER_MAIN_PATH)
assert signature_silver(apres_optimize) == signature_avant, "Les lignes Silver doivent être conservées."
fichiers_apres = DeltaTable.forPath(spark, SILVER_MAIN_PATH).detail().select("numFiles").first().numFiles
print(f"Compaction Silver : {fichiers_avant} fichier(s) avant, {fichiers_apres} après ; contenu vérifié.")

Tranches de prix au taux de commentaire le plus élevé, pour chaque niveau de retard (au moins 100 commandes)
+---------------+---------------+------------+---------------+-------------------+
|tranche_retard |tranche_prix   |nb_commandes|nb_commentaires|taux_commentaire   |
+---------------+---------------+------------+---------------+-------------------+
|À temps        |Plus de 200 BRL|10635       |4359           |0.40987306064880114|
|1 à 3 jours    |0 à 20 BRL     |122         |46             |0.3770491803278688 |
|4 à 7 jours    |]100, 200] BRL |433         |42             |0.09699769053117784|
|8 jours et plus|Plus de 200 BRL|459         |9              |0.0196078431372549 |
+---------------+---------------+------------+---------------+-------------------+

Les autres tranches sont aussi enregistrées dans la table Gold pour comparer tous les prix à retard identique.
[GOLD OK] /lakehouse/gold/olist_commentaires_categories : 74 lignes agrégées
[GOLD OK] /lakehouse/gold/olist_commen